# 0.11 — Generative AI: keyword timeline + snowball lexicon

**Question:** when does Bloomberg all-news start talking about **generative AI** — and what vocabulary emerges over time?

Complements the unsupervised path [`0.9`](0.9-genai-unsupervised-detection.ipynb) / [`0.10`](0.10-genai-hierarchy-drilldown.ipynb) with an **explicit, auditable keyword layer** (validation / labeling, not discovery input).

**Protocol:**
1. **Seed lexicon** — curated genAI / LLM / infra terms (regex, word boundaries).
2. **Snowball expansion** — iteratively mine 1–2 grams from seed-matched headlines; keep terms with high **lift** vs the full corpus; block generic finance boilerplate.
3. **Timeline** — weekly counts + share of all headlines; overlay ChatGPT launch and CHAT ETF inception.

**Milestones (annotation only):** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

**Outputs:** `genai_seed_timeline.parquet`, `genai_snowball_lexicon.parquet`, `genai_snowball_timeline.parquet`, `genai_keyword_timeline.html`, `genai_first_seen.parquet`, `genai_lexicon_monthly_2018.parquet`, `genai_lexicon_timeline_2018.html`

Burst-window BERTrend (Nov 2022 – Jan 2023) → [`0.12`](0.12-genai-burst-bertrend.ipynb)

In [1]:
import json, lzma, re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- window (matches 0.9 / cached meta; extend YEARS + DATE_* for earlier raw years) ---
YEARS = [2021, 2022, 2023]
DATE_START, DATE_END = pd.Timestamp("2021-01-01"), pd.Timestamp("2023-12-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")  # CHAT ETF
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
META_CACHE = OUTPUT_DIR / "genai_full_meta.parquet"
RANDOM_SEED = 42

FREQ = "W-MON"          # weekly buckets (Monday start)
SNOWBALL_ROUNDS = 3
SNOWBALL_TOP_K = 25     # new terms per round
MIN_TERM_HITS = 15      # min occurrences in seed-matched headlines
MIN_LIFT = 3.0            # P(term|hit) / P(term|all)
MAX_LEXICON = 120

FINANCE_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock", "stocks",
    "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan", "bn",
    "march", "april", "june", "july", "august", "september", "october", "november", "december",
}

# Seed phrases (matched as substring / regex) + single-token seeds (word boundary)
SEED_PHRASES = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b", r"\bllm\b",
    r"chatgpt", r"gpt-3", r"gpt-4", r"gpt-3\.5",
    r"openai", r"anthropic", r"\bclaude\b", r"\bgemini\b", r"\bbard\b",
    r"copilot", r"midjourney", r"stable diffusion", r"dall-e", r"dalle",
    r"prompt engineering", r"foundation model", r"foundation models",
    r"text-to-image", r"text to image", r"ai chatbot", r"ai chat bot",
]
SEED_WORDS = [
    "openai", "chatgpt", "anthropic", "midjourney", "copilot", "gemini",
]

print(f"Window {DATE_START.date()}→{DATE_END.date()} | snowball {SNOWBALL_ROUNDS} rounds")

Window 2021-01-01→2023-12-31 | snowball 3 rounds


## 1. Load headlines

Uses `genai_full_meta.parquet` from [`0.10`](0.10-genai-hierarchy-drilldown.ipynb) when present; otherwise loads raw Bloomberg wires.

In [2]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text

if META_CACHE.exists():
    news = pd.read_parquet(META_CACHE)
    news["date"] = pd.to_datetime(news["date"])
    news = news[(news.date >= DATE_START) & (news.date <= DATE_END)].reset_index(drop=True)
    print(f"Loaded {META_CACHE.name}: {len(news):,} headlines")
else:
    frames = []
    for yr in YEARS:
        path = RAW_DIR / f"raw_news_{yr}.csv.xz"
        if not path.exists():
            raise FileNotFoundError(f"Missing {path} — run 0.10 embed cache or add raw years")
        with lzma.open(path, "rb") as f:
            part = (
                pl.scan_csv(f, infer_schema_length=10_000)
                .select(["Headline", "CaptureTime", "WireName"])
                .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
                .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
                .collect()
            )
        frames.append(part)
        print(f"{yr}: {part.height:>9,} Bloomberg rows")
    news = pl.concat(frames).to_pandas()
    news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
    news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
    news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
    news["Headline"] = news["Headline"].map(strip_prefix)
    news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)
    print(f"\nTotal Bloomberg headlines: {len(news):,}")

news["headline_lc"] = news["Headline"].str.lower()
news["week"] = news["date"].dt.to_period(FREQ).dt.start_time

Loaded genai_full_meta.parquet: 2,954,113 headlines


## 2. Matching helpers

In [3]:
TOKEN_RE = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\b")
BIGRAM_RE = re.compile(r"(?u)\b[a-z][a-z0-9\-]{2,}\s+[a-z][a-z0-9\-]{2,}\b")


def compile_lexicon(phrases: list[str], words: list[str]) -> re.Pattern:
    parts = list(phrases)
    for w in words:
        w = w.strip().lower()
        if w and w not in FINANCE_STOP:
            parts.append(rf"\b{re.escape(w)}\b")
    if not parts:
        return re.compile(r"(?!)")
    return re.compile("|".join(f"(?:{p})" for p in parts), re.I)


def match_mask(headlines: pd.Series, pattern: re.Pattern) -> pd.Series:
    return headlines.str.contains(pattern, na=False, regex=True)


def extract_ngrams(texts: pd.Series) -> Counter:
    ctr = Counter()
    for text in texts:
        t = text.lower()
        for bg in BIGRAM_RE.findall(t):
            if all(tok not in FINANCE_STOP for tok in bg.split()):
                ctr[bg] += 1
        for tok in TOKEN_RE.findall(t):
            if tok not in FINANCE_STOP and len(tok) >= 3:
                ctr[tok] += 1
    return ctr


def snowball_candidates(hit_texts: pd.Series, bg_texts: pd.Series,
                        existing: set[str], top_k: int = SNOWBALL_TOP_K) -> list[tuple[str, float, int]]:
    hit_ctr = extract_ngrams(hit_texts)
    bg_ctr = extract_ngrams(bg_texts)
    n_hit, n_bg = len(hit_texts), max(len(bg_texts), 1)
    scored = []
    for term, c_hit in hit_ctr.items():
        if term in existing or c_hit < MIN_TERM_HITS:
            continue
        c_bg = bg_ctr.get(term, 0)
        p_hit = c_hit / n_hit
        p_bg = (c_bg + 1) / n_bg
        lift = p_hit / p_bg
        if lift >= MIN_LIFT:
            scored.append((term, lift, c_hit))
    scored.sort(key=lambda x: (-x[1], -x[2]))
    return scored[:top_k]


def weekly_timeline(df: pd.DataFrame, label: str) -> pd.DataFrame:
    weekly = df.groupby("week").size().rename("hits").reset_index()
    totals = news.groupby("week").size().rename("total").reset_index()
    out = totals.merge(weekly, on="week", how="left").fillna({"hits": 0})
    out["hits"] = out["hits"].astype(int)
    out["share_bp"] = (out["hits"] / out["total"] * 10_000).round(2)
    out["lexicon"] = label
    return out


def first_seen_table(df: pd.DataFrame, terms: list[str]) -> pd.DataFrame:
    rows = []
    for term in terms:
        if " " in term:
            pat = re.compile(re.escape(term), re.I)
        else:
            pat = re.compile(rf"\b{re.escape(term)}\b", re.I)
        m = match_mask(df["headline_lc"], pat)
        if not m.any():
            continue
        first = df.loc[m, "date"].min()
        rows.append({"term": term, "first_seen": first, "hits": int(m.sum())})
    return pd.DataFrame(rows).sort_values("first_seen").reset_index(drop=True)


seed_pat = compile_lexicon(SEED_PHRASES, SEED_WORDS)
news["seed_hit"] = match_mask(news["headline_lc"], seed_pat)
print(f"Seed lexicon: {len(SEED_PHRASES)} phrases + {len(SEED_WORDS)} words")
print(f"Seed hits: {news['seed_hit'].sum():,} / {len(news):,} "
      f"({news['seed_hit'].mean()*100:.3f}%)")
if news["seed_hit"].any():
    print(f"First seed headline: {news.loc[news['seed_hit'], 'date'].min().date()}")
    print(f"Last  seed headline: {news.loc[news['seed_hit'], 'date'].max().date()}")

Seed lexicon: 27 phrases + 6 words
Seed hits: 1,909 / 2,954,113 (0.065%)
First seed headline: 2021-01-04
Last  seed headline: 2023-12-30


## 3. Seed lexicon timeline

In [4]:
seed_hits = news[news["seed_hit"]].copy()
seed_timeline = weekly_timeline(seed_hits, "seed")
seed_timeline.to_parquet(OUTPUT_DIR / "genai_seed_timeline.parquet", index=False)

seed_by_term = first_seen_table(news, [
    "chatgpt", "openai", "generative ai", "large language model", "llm",
    "anthropic", "claude", "gemini", "copilot", "gpt-4", "gpt-3",
])
print("First-seen (selected seed terms):")
print(seed_by_term.to_string(index=False))

print("\nSample seed headlines (first 5 chronologically):")
for _, r in seed_hits.sort_values("date").head(5).iterrows():
    print(f"  {r.date.date()}  {r.Headline[:100]}")

print("\nSample seed headlines (Nov 2022 – Mar 2023):")
mid = seed_hits[(seed_hits.date >= "2022-11-01") & (seed_hits.date <= "2023-03-31")]
for _, r in mid.sort_values("date").head(8).iterrows():
    print(f"  {r.date.date()}  {r.Headline[:100]}")

First-seen (selected seed terms):
                term              first_seen  hits
              gemini 2021-01-04 05:06:11.318   263
              claude 2021-01-11 10:21:48.578    21
              openai 2021-05-07 20:08:35.399   744
           anthropic 2022-04-29 18:09:11.371    72
             copilot 2022-11-02 13:34:00.144    14
             chatgpt 2022-12-07 05:00:15.903   524
       generative ai 2023-02-01 14:01:42.246   224
large language model 2023-02-24 16:11:53.115    11
               gpt-4 2023-03-14 17:00:41.584    18
                 llm 2023-05-03 22:16:52.452     3

Sample seed headlines (first 5 chronologically):
  2021-01-04  *SASO COMPLETES DIVESTMENT OF 50% INTEREST IN GEMINI HDPE LLC
  2021-01-04  Sasol Completes Divestment of 50% Interest in Gemini
  2021-01-04  *ION ANNOUNCES COMMERCIAL DEPLOYMENT OF GEMINI FOR A SUPER MAJOR
  2021-01-04  *ION SHARES CLIMB 18% AFTER ANNOUNCEMENT REGARDING GEMINI
  2021-01-04  Ion Shares Climb 25% After Announcement Regardi

## 4. Snowball lexicon expansion

Iteratively add high-lift n-grams from seed-matched headlines. Terms already in the lexicon are excluded; generic finance tokens are blocked via `FINANCE_STOP`.

In [5]:
lexicon_phrases = list(SEED_PHRASES)
lexicon_words = list(SEED_WORDS)
lexicon_terms = set()
expansion_log = []

for rnd in range(1, SNOWBALL_ROUNDS + 1):
    pat = compile_lexicon(lexicon_phrases, lexicon_words)
    hit_mask = match_mask(news["headline_lc"], pat)
    hit_texts = news.loc[hit_mask, "headline_lc"]
    bg_texts = news.loc[~hit_mask, "headline_lc"]
    existing = lexicon_terms | {t.lower() for t in lexicon_words}
    cands = snowball_candidates(hit_texts, bg_texts, existing, top_k=SNOWBALL_TOP_K)
    added = []
    for term, lift, cnt in cands:
        if len(lexicon_terms) >= MAX_LEXICON:
            break
        tl = term.lower()
        if tl in lexicon_terms:
            continue
        lexicon_terms.add(tl)
        if " " in term:
            lexicon_phrases.append(re.escape(term))
        else:
            lexicon_words.append(term)
        added.append({"round": rnd, "term": term, "lift": round(lift, 2), "hits_in_seed": cnt})
    expansion_log.extend(added)
    print(f"Round {rnd}: +{len(added)} terms (lexicon size {len(lexicon_terms) + len(SEED_PHRASES)})")
    for row in added[:8]:
        print(f"  + {row['term']:<28} lift={row['lift']:>6.1f}  hits={row['hits_in_seed']}")
    if not added:
        print("  (converged — no new terms)")
        break

expansion_df = pd.DataFrame(expansion_log)
expansion_df.to_parquet(OUTPUT_DIR / "genai_snowball_lexicon.parquet", index=False)
print(f"\nSnowball added {len(expansion_df)} terms → saved genai_snowball_lexicon.parquet")
if not expansion_df.empty:
    print(expansion_df.head(20).to_string(index=False))

Round 1: +25 terms (lexicon size 52)
  + bard                         lift=74230.4  hits=48
  + gemini therapeutics          lift=63405.1  hits=41
  + chatgpt-like                 lift=63405.1  hits=41
  + openai board                 lift=60312.2  hits=39
  + startup anthropic            lift=37115.2  hits=24
  + gpt-4                        lift=27836.4  hits=18
  + openai staff                 lift=26289.9  hits=17
  + claude                       lift=24743.5  hits=16
Round 2: +25 terms (lexicon size 77)
  + sam bankman-fried            lift=160497.5  hits=179
  + sam fazeli                   lift=36762.0  hits=41
  + sam zell                     lift=16139.4  hits=18
  + brockman                     lift=1517.4  hits=22
  + silbert                      lift= 996.3  hits=20
  + fazeli                       lift= 854.9  hits=41
  + zell                         lift= 415.5  hits=19
  + nadella                      lift= 393.6  hits=18
Round 3: +25 terms (lexicon size 102)
  + google 

## 5. Snowball timeline + comparison

In [6]:
snow_pat = compile_lexicon(lexicon_phrases, lexicon_words)
news["snow_hit"] = match_mask(news["headline_lc"], snow_pat)
snow_hits = news[news["snow_hit"]].copy()
snow_timeline = weekly_timeline(snow_hits, "snowball")
snow_timeline.to_parquet(OUTPUT_DIR / "genai_snowball_timeline.parquet", index=False)

compare = seed_timeline.merge(
    snow_timeline[["week", "hits", "share_bp"]].rename(
        columns={"hits": "snow_hits", "share_bp": "snow_share_bp"}),
    on="week", how="left",
).rename(columns={"hits": "seed_hits", "share_bp": "seed_share_bp"})

print(f"Snowball hits: {news['snow_hit'].sum():,} ({news['snow_hit'].mean()*100:.3f}% of corpus)")
print(f"Incremental:   {(news['snow_hit'] & ~news['seed_hit']).sum():,} headlines beyond seed")

pre_chatgpt = news[news.date < CHATGPT_LAUNCH]
pre_chat = news[news.date < INCEPTION]
print(f"\nPre-ChatGPT ({CHATGPT_LAUNCH.date()}): seed {pre_chatgpt['seed_hit'].sum():,} | snowball {pre_chatgpt['snow_hit'].sum():,}")
print(f"Pre-CHAT ETF  ({INCEPTION.date()}):     seed {pre_chat['seed_hit'].sum():,} | snowball {pre_chat['snow_hit'].sum():,}")

first_snow = first_seen_table(news, sorted(lexicon_terms)[:40])
first_snow.to_parquet(OUTPUT_DIR / "genai_first_seen.parquet", index=False)
print("\nFirst-seen snowball terms (subset):")
print(first_snow.head(15).to_string(index=False))

Snowball hits: 24,012 (0.813% of corpus)
Incremental:   22,103 headlines beyond seed

Pre-ChatGPT (2022-11-30): seed 176 | snowball 14,050
Pre-CHAT ETF  (2023-05-17):     seed 834 | snowball 18,641

First-seen snowball terms (subset):
         term              first_seen  hits
         club 2021-01-01 06:54:11.110  1164
       fazeli 2021-01-03 11:09:17.340    83
          bot 2021-01-03 22:32:32.507  1138
      bahamas 2021-01-03 22:37:48.068   272
         bing 2021-01-03 22:38:15.114    88
         apps 2021-01-03 22:46:41.469   586
       google 2021-01-04 07:24:12.020  4169
illinois tool 2021-01-04 15:09:11.497   161
     brockman 2021-01-04 22:19:12.274    34
      genesis 2021-01-05 05:35:19.885   680
   developing 2021-01-05 13:46:08.387   612
 enerpac tool 2021-01-07 08:01:19.692    83
        baidu 2021-01-07 08:04:52.698   848
       claude 2021-01-11 10:21:48.578    21
   baidu adrs 2021-01-13 16:23:07.218    50


## 6. Plot weekly intensity

In [7]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Weekly headline count", "Share of all headlines (basis points)"))

for col, name, color in [
    ("seed_hits", "Seed lexicon", "#636EFA"),
    ("snow_hits", "Snowball lexicon", "#EF553B"),
]:
    fig.add_trace(go.Scatter(x=compare.week, y=compare[col], name=name,
                             line=dict(color=color, width=1.5)), row=1, col=1)

for col, name, color in [
    ("seed_share_bp", "Seed share (bp)", "#636EFA"),
    ("snow_share_bp", "Snowball share (bp)", "#EF553B"),
]:
    fig.add_trace(go.Scatter(x=compare.week, y=compare[col], name=name,
                             line=dict(color=color, width=1.5), showlegend=False), row=2, col=1)

for dt, label in [(CHATGPT_LAUNCH, "ChatGPT"), (INCEPTION, "CHAT ETF")]:
    for row in (1, 2):
        fig.add_vline(x=dt, line_width=1, line_dash="dash", line_color="gray", row=row, col=1)
    fig.add_annotation(x=dt, y=1.02, yref="paper", text=label, showarrow=False,
                       font=dict(size=10, color="gray"), xanchor="center")

fig.update_layout(height=620, title="Generative AI keyword intensity (Bloomberg all-news)",
                  legend=dict(orientation="h", y=1.12), hovermode="x unified")
fig.update_yaxes(title_text="Headlines / week", row=1, col=1)
fig.update_yaxes(title_text="Basis points", row=2, col=1)

html_path = OUTPUT_DIR / "genai_keyword_timeline.html"
fig.write_html(html_path)
fig.show()
print(f"Saved {html_path.name}")

Saved genai_keyword_timeline.html


## 7. Monthly rollup + headline samples by era

In [8]:
news["month"] = news["date"].dt.to_period("M").dt.start_time
monthly = (
    news.groupby("month")
    .agg(total=("Headline", "count"), seed=("seed_hit", "sum"), snow=("snow_hit", "sum"))
    .reset_index()
)
monthly["seed_share_bp"] = (monthly["seed"] / monthly["total"] * 10_000).round(2)
monthly["snow_share_bp"] = (monthly["snow"] / monthly["total"] * 10_000).round(2)
monthly.to_parquet(OUTPUT_DIR / "genai_keyword_monthly.parquet", index=False)
print("Monthly rollup (last 24 months):")
print(monthly.tail(24).to_string(index=False))

ERAS = [
    ("2021 H1", "2021-01-01", "2021-06-30"),
    ("2021 H2", "2021-07-01", "2021-12-31"),
    ("2022 H1", "2022-01-01", "2022-06-30"),
    ("2022 pre-ChatGPT", "2022-07-01", "2022-11-29"),
    ("2022 post-ChatGPT", "2022-11-30", "2022-12-31"),
    ("2023 pre-CHAT", "2023-01-01", "2023-05-16"),
    ("2023 post-CHAT", "2023-05-17", "2023-12-31"),
]
print("\nEra summary (snowball lexicon):")
for label, t0, t1 in ERAS:
    sub = news[(news.date >= t0) & (news.date <= t1)]
    n, h = len(sub), int(sub["snow_hit"].sum())
    print(f"  {label:<20}  {h:>5,} / {n:>8,}  ({h/max(n,1)*100:.3f}%)")

print("\nRandom snowball samples — 2023 pre-CHAT (n=6):")
pre = snow_hits[(snow_hits.date >= "2023-01-01") & (snow_hits.date < INCEPTION)]
for _, r in pre.sample(min(6, len(pre)), random_state=RANDOM_SEED).iterrows():
    print(f"  {r.date.date()}  {r.Headline[:95]}")

Monthly rollup (last 24 months):
     month  total  seed  snow  seed_share_bp  snow_share_bp
2022-01-01  75484     6   653           0.79          86.51
2022-02-01  90578    13   532           1.44          58.73
2022-03-01  90330     7   605           0.77          66.98
2022-04-01  83620     5   484           0.60          57.88
2022-05-01  97068     3   550           0.31          56.66
2022-06-01  73737     9   628           1.22          85.17
2022-07-01  77278     6   680           0.78          87.99
2022-08-01  87360    16   456           1.83          52.20
2022-09-01  67601     4   492           0.59          72.78
2022-10-01  81113     5   572           0.62          70.52
2022-11-01  91946    23  1441           2.50         156.72
2022-12-01  56600    27   921           4.77         162.72
2023-01-01  70224   112   931          15.95         132.58
2023-02-01  86888   163   812          18.76          93.45
2023-03-01  82524   126   713          15.27          86.40
2023-04

## 8. Best genAI lexicon + monthly timeline (2018+)

Tiered, disambiguated keywords (no bare `gemini`/`claude`, no snowball). Loads **2018–2023** Bloomberg wires from raw (cached). Run **§0**, then this cell.

In [9]:
# --- best genAI lexicon (validation layer) ---
GENAI_T1_STRICT = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle", r"text-to-image", r"text to image",
]
GENAI_T2_STANDARD = [
    r"\bopenai\b", r"\banthropic\b",
    r"chatgpt-like", r"chatgpt-style", r"prompt engineering",
    r"ai chatbot", r"ai chat bot",
]
# Tier 3: token + co-occurring genAI anchor in same headline
GENAI_T3_GATED = [
    (r"\bclaude\b", r"anthropic|chatgpt|openai|generative|\bllm\b|language model"),
    (r"\bgemini\b", r"google|\bbard\b|generative|chatgpt|openai|\bllm\b"),
    (r"\bbard\b", r"google|generative|chatgpt|openai|\bllm\b"),
    (r"\bcopilot\b", r"microsoft|github|generative|chatgpt|openai|\bllm\b"),
    (r"\bllm\b", r"openai|anthropic|chatgpt|generative|language model|foundation model"),
]

LEXICON_YEARS = list(range(2018, 2024))
LEXICON_START, LEXICON_END = pd.Timestamp("2018-01-01"), pd.Timestamp("2023-12-31")
HEADLINES_2018_CACHE = OUTPUT_DIR / "bloomberg_headlines_2018_2023.parquet"


def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text


def _compile_or(phrases: list[str]) -> re.Pattern:
    return re.compile("|".join(f"(?:{p})" for p in phrases), re.I)


def _match_t1_t2(hl: pd.Series, phrases: list[str]) -> pd.Series:
    if not phrases:
        return pd.Series(False, index=hl.index)
    return hl.str.contains(_compile_or(phrases), na=False, regex=True)


def _match_gated(hl: pd.Series, gated: list[tuple[str, str]]) -> pd.Series:
    out = pd.Series(False, index=hl.index)
    for tok_pat, ctx_pat in gated:
        tok = re.compile(tok_pat, re.I)
        ctx = re.compile(ctx_pat, re.I)
        out |= hl.str.contains(tok, na=False) & hl.str.contains(ctx, na=False)
    return out


def genai_masks(hl: pd.Series) -> dict[str, pd.Series]:
    strict = _match_t1_t2(hl, GENAI_T1_STRICT)
    standard = strict | _match_t1_t2(hl, GENAI_T2_STANDARD)
    broad = standard | _match_gated(hl, GENAI_T3_GATED)
    return {"strict": strict, "standard": standard, "broad": broad}


def load_bloomberg_headlines(years: list[int], date_start, date_end) -> pd.DataFrame:
    frames = []
    for yr in years:
        path = RAW_DIR / f"raw_news_{yr}.csv.xz"
        if not path.exists():
            raise FileNotFoundError(f"Missing {path}")
        with lzma.open(path, "rb") as f:
            part = (
                pl.scan_csv(f, infer_schema_length=10_000)
                .select(["Headline", "CaptureTime", "WireName"])
                .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
                .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
                .collect()
            )
        frames.append(part)
        print(f"  {yr}: {part.height:>9,} Bloomberg rows")
    df = pl.concat(frames).to_pandas()
    df["date"] = pd.to_datetime(df["CaptureTime"]).dt.tz_localize(None)
    df = df[(df.date >= date_start) & (df.date <= date_end)]
    df = df.dropna(subset=["Headline"]).drop_duplicates("Headline")
    df["Headline"] = df["Headline"].map(strip_prefix)
    df = df[df.Headline.str.split().map(len) >= 4].reset_index(drop=True)
    return df


if HEADLINES_2018_CACHE.exists():
    news18 = pd.read_parquet(HEADLINES_2018_CACHE)
    news18["date"] = pd.to_datetime(news18["date"])
    print(f"Loaded cache {HEADLINES_2018_CACHE.name}: {len(news18):,} headlines")
else:
    print(f"Loading raw Bloomberg {LEXICON_START.date()}→{LEXICON_END.date()} (one-time, ~10 min)…")
    news18 = load_bloomberg_headlines(LEXICON_YEARS, LEXICON_START, LEXICON_END)
    news18.to_parquet(HEADLINES_2018_CACHE, index=False)
    print(f"Wrote {HEADLINES_2018_CACHE.name}: {len(news18):,} headlines")

news18["hl"] = news18["Headline"].str.lower()
news18["month"] = news18["date"].dt.to_period("M").dt.start_time
masks18 = genai_masks(news18["hl"])

print("\nLexicon definitions:")
print(f"  STRICT   (T1): {len(GENAI_T1_STRICT)} phrases — core genAI vocabulary")
print(f"  STANDARD (T1+T2): +{len(GENAI_T2_STANDARD)} patterns — add OpenAI / Anthropic / chatbot")
print(f"  BROAD    (+T3 gated): +{len(GENAI_T3_GATED)} context rules — Claude/Gemini/Copilot/LLM")

print("\nCorpus totals by lexicon:")
for name, m in masks18.items():
    pre_cgpt = m & (news18.date < CHATGPT_LAUNCH)
    pre_chat = m & (news18.date < INCEPTION)
    print(f"  {name:8}  all={int(m.sum()):>6,}  pre-ChatGPT={int(pre_cgpt.sum()):>5,}  pre-CHAT={int(pre_chat.sum()):>5,}")

monthly18 = news18.groupby("month").size().rename("total").reset_index()
for name, m in masks18.items():
    hits = news18.loc[m].groupby("month").size().rename(name).reset_index()
    monthly18 = monthly18.merge(hits, on="month", how="left")
monthly18 = monthly18.fillna(0)
for name in masks18:
    monthly18[name] = monthly18[name].astype(int)
    monthly18[f"{name}_share_bp"] = (monthly18[name] / monthly18["total"] * 10_000).round(2)

monthly18.to_parquet(OUTPUT_DIR / "genai_lexicon_monthly_2018.parquet", index=False)
print("\nMonthly hits (selected):")
show = monthly18[(monthly18.month >= "2018-01-01") & (monthly18.month <= "2023-12-01")]
print(show[show.month.dt.month.isin([1, 6, 11, 12])].tail(24).to_string(index=False))

fig18 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                      subplot_titles=("Monthly genAI headline count (2018–2023)",
                                      "Share of all headlines (basis points)"))
colors = {"strict": "#636EFA", "standard": "#EF553B", "broad": "#00CC96"}
for name in ("strict", "standard", "broad"):
    fig18.add_trace(go.Scatter(x=monthly18.month, y=monthly18[name], name=name.title(),
                               line=dict(color=colors[name], width=1.5)), row=1, col=1)
    fig18.add_trace(go.Scatter(x=monthly18.month, y=monthly18[f"{name}_share_bp"],
                               name=f"{name.title()} share", showlegend=False,
                               line=dict(color=colors[name], width=1.2, dash="dot")), row=2, col=1)
for dt, label in [(CHATGPT_LAUNCH, "ChatGPT"), (INCEPTION, "CHAT ETF")]:
    for row in (1, 2):
        fig18.add_vline(x=dt, line_width=1, line_dash="dash", line_color="gray", row=row, col=1)
fig18.update_layout(height=650, title="GenAI lexicon intensity from 2018", hovermode="x unified",
                    legend=dict(orientation="h", y=1.1))
fig18.update_yaxes(title_text="Headlines / month", row=1, col=1)
fig18.update_yaxes(title_text="Basis points", row=2, col=1)
path18 = OUTPUT_DIR / "genai_lexicon_timeline_2018.html"
fig18.write_html(path18)
fig18.show()
print(f"\nSaved genai_lexicon_monthly_2018.parquet + {path18.name}")

Loaded cache bloomberg_headlines_2018_2023.parquet: 8,658,509 headlines

Lexicon definitions:
  STRICT   (T1): 17 phrases — core genAI vocabulary
  STANDARD (T1+T2): +7 patterns — add OpenAI / Anthropic / chatbot
  BROAD    (+T3 gated): +5 context rules — Claude/Gemini/Copilot/LLM

Corpus totals by lexicon:
  strict    all=   797  pre-ChatGPT=    8  pre-CHAT=  400
  standard  all= 1,583  pre-ChatGPT=   27  pre-CHAT=  590
  broad     all= 1,619  pre-ChatGPT=   30  pre-CHAT=  605

Monthly hits (selected):
     month  total  strict  standard  broad  strict_share_bp  standard_share_bp  broad_share_bp
2018-01-01 194061       0         0      0             0.00               0.00            0.00
2018-06-01 172944       1         1      3             0.06               0.06            0.17
2018-11-01 130216       0         0      0             0.00               0.00            0.00
2018-12-01 105404       0         0      0             0.00               0.00            0.00
2019-01-01 12432


Saved genai_lexicon_monthly_2018.parquet + genai_lexicon_timeline_2018.html


## 9. Quick reload (parquet + chart only)

Run §0, then this cell.

In [ ]:
# See [`0.12`](0.12-genai-burst-bertrend.ipynb) — BERTrend on Nov 2022–Jan 2023 burst window.

In [ ]:
# 1. Supervised: vocabulary definition
# 2. Semi-supervised con soft clustering, posso avere genAI come combinazione lineare dei vettori dei topic estratti in precedenza
# 3. Fully unsupervised